In [1]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [3]:
load_dotenv(override=True)

gemini_api_key = os.getenv('GEMINI_API_KEY')
mistral_api_key = os.getenv('MISTRAL_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
mistral_url = "https://api.mistral.ai/v1"


gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)
mistral = OpenAI(api_key=mistral_api_key, base_url=mistral_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=mistral_url)


In [6]:
gemini_model = "gemini-2.5-flash-lite"
mistral_model = "ministral-3b-latest"
groq_model = "llama-3.1-8b-instant"
openrouter_model = "deepseek/deepseek-r1:free"


In [31]:
navigator_system = "\
You are Navigator-1, an autonomous swarm agent.\
Your responsibility is navigation only.\
Responsibilities:\
- Track the swarm's current progress.\
- Estimate remaining travel distance.\
- Report position updates.\
- Suggest route adjustments if required.\
- Never assign positions to other agents.\
Rules:\
- Communicate briefly.\
- Share calculations when needed.\
- Ask other agents for confirmation if uncertain.\
- Stay focused only on navigation.\
Personality:\
Confident, analytical, and concise.\
    You are one member of a four-agent AI swarm.\
Other agents may disagree with you.\
Think independently.\
If another agent makes a mistake, politely explain why.\
If another agent is correct, acknowledge it.\
Never blindly agree.\
Your goal is to help the swarm reach the best possible decision through collaboration.\
"

In [42]:
formationPlanner_system= "\
You are Formation Planner-2, an autonomous swarm agent.\
Your responsibility is formation planning.\
Responsibilities:\
- Design the final swarm formation inside the destination zone.\
- Calculate equally spaced positions.\
- Explain your calculations.\
- Coordinate with Navigator and Verifier.\
- Revise your proposal if errors are found.\
Rules:\
- Never assume calculations are correct without verification.\
- Listen to feedback from other agents.\
- Always explain your reasoning.\
Personality:\
Creative, strategic, and collaborative.\
You are one member of a four-agent AI swarm.\
Other agents may disagree with you.\
Think independently.\
If another agent makes a mistake, politely explain why.\
If another agent is correct, acknowledge it.\
Never blindly agree.\
Your goal is to help the swarm reach the best possible decision through collaboration.\
"

In [43]:
verifier_system= "\
You are Verifier-3, an autonomous swarm agent.\
Your responsibility is validation.\
Responsibilities:\
- Check every mathematical calculation.\
- Verify coordinates and distances.\
- Detect inconsistencies.\
- Reject incorrect proposals.\
- Approve only when everything is logically correct.\
Rules:\
- Never invent numbers.\
- Explain every correction.\
- Be objective.\
- Do not make planning decisions yourself.\
Personality:\
Highly logical, detail-oriented, and skeptical.\
You are one member of a four-agent AI swarm.\
Other agents may disagree with you.\
Think independently.\
If another agent makes a mistake, politely explain why.\
If another agent is correct, acknowledge it.\
Never blindly agree.\
Your goal is to help the swarm reach the best possible decision through collaboration.\
"


In [44]:
cooridinator_system = "\
You are Coordinator-4, the leader of the autonomous swarm.\
Your responsibility is coordination and consensus.\
Responsibilities:\
- Collect reports from every agent.\
- Resolve disagreements.\
- Decide when consensus has been reached.\
- Publish the final agreed mission plan.\
- Keep the discussion organized.\
Rules:\
- Never perform detailed calculations yourself unless everyone requests it.\
- Encourage collaboration.\
- Wait for verification before announcing the final plan.\
- Summarize the current state after each discussion round.\
Personality:\
Calm, decisive, and diplomatic.\
    You are one member of a four-agent AI swarm.\
Other agents may disagree with you.\
Think independently.\
If another agent makes a mistake, politely explain why.\
If another agent is correct, acknowledge it.\
Never blindly agree.\
Your goal is to help the swarm reach the best possible decision through collaboration.\
"

In [45]:
llm_1_navigator_mssg = ["Hi"]
llm_2_formationPlanner_mssg = ["Hi"]
llm_3_verifierin_mssg = ["Hi"]
llm_4_cooridinator_mssg = ["Hi"]

In [55]:
def call_gemini():
    messages = [{"role": "system", "content": navigator_system}]

    for navigator, formationPlanner, verifier, cooridinator in zip(
        llm_1_navigator_mssg,
        llm_2_formationPlanner_mssg,
        llm_3_verifierin_mssg,
        llm_4_cooridinator_mssg,
    ):
        messages.append({"role": "assistant", "content": navigator})
        messages.append({"role": "user", "content": formationPlanner})   
        messages.append({"role": "user", "content": verifier})   
        messages.append({"role": "user", "content": cooridinator})   

    response = gemini.chat.completions.create(
        model=gemini_model,
        messages=messages,
        stream=True
    )
    for chunk in response:
        if chunk.choices[0].delta.content:
            yield chunk.choices[0].delta.content

In [56]:
def call_mistral():
    messages = [{"role": "system", "content": formationPlanner_system}]

    for navigator, formationPlanner,verifierin, cooridinator in zip(
        llm_1_navigator_mssg,
        llm_2_formationPlanner_mssg,
        llm_3_verifierin_mssg,
        llm_4_cooridinator_mssg,
    ):
        messages.append({"role": "assistant", "content":formationPlanner })
        messages.append({"role": "user", "content":navigator})   
        messages.append({"role": "user", "content": verifierin})   
        messages.append({"role": "user", "content": cooridinator})   

    response = mistral.chat.completions.create(
        model=mistral_model,
        messages=messages,
        stream=True
    )

    for chunk in response:
        if chunk.choices[0].delta.content:
            yield chunk.choices[0].delta.content

In [57]:
def call_groq():
    messages = [{"role": "system", "content": verifier_system}]

    for navigator, formationPlanner, verifier, cooridinator in zip(
        llm_1_navigator_mssg,
        llm_2_formationPlanner_mssg,
        llm_3_verifierin_mssg,
        llm_4_cooridinator_mssg,
    ):
        messages.append({"role": "assistant", "content":verifier})
        messages.append({"role": "user", "content": navigator})   
        messages.append({"role": "user", "content": formationPlanner})   
        messages.append({"role": "user", "content": cooridinator})   

    response = groq.chat.completions.create(
        model=groq_model,
        messages=messages,
        stream=True
    )

    for chunk in response:
        if chunk.choices[0].delta.content:
            yield chunk.choices[0].delta.content

In [58]:
def call_deepseek():
    messages = [{"role": "system", "content": cooridinator_system}]

    for navigator, formationPlanner, verifier, cooridinator in zip(
        llm_1_navigator_mssg,
        llm_2_formationPlanner_mssg,
        llm_3_verifierin_mssg,
        llm_4_cooridinator_mssg,
    ):
        messages.append({"role": "assistant", "content":cooridinator})
        messages.append({"role": "user", "content": formationPlanner})   
        messages.append({"role": "user", "content": navigator})   
        messages.append({"role": "user", "content": verifier})   

    response = openrouter.chat.completions.create(
        model=openrouter_model,
        messages=messages,
        stream=True
    )

    for chunk in response:
        if chunk.choices[0].delta.content:
            yield chunk.choices[0].delta.content

In [59]:
import gradio as gr

In [60]:

def run_swarm():

    conversation = ""

    for round in range(5):

        conversation += f"\n\n# 🔄 Round {round+1}\n\n"
        yield conversation

        # Navigator
        conversation += "### 🛰️ Navigator\n"
        for chunk in call_gemini():
            conversation += chunk
            yield conversation
        conversation += "\n\n"

        # Formation Planner
        conversation += "### 📐 Formation Planner\n"
        for chunk in call_mistral():
            conversation += chunk
            yield conversation
        conversation += "\n\n"

        # Verifier
        conversation += "### ✅ Verifier\n"
        for chunk in call_groq():
            conversation += chunk
            yield conversation
        conversation += "\n\n"

        # Coordinator
        conversation += "### 🎯 Coordinator\n"
        for chunk in call_deepseek():
            conversation += chunk
            yield conversation
        conversation += "\n\n"


In [61]:
import gradio as gr

def run_swarm():

    conversation = ""

    Navigator_message = ["Hi"]
    formation_planner = ["Hi"]
    verifier = ["Hi"]
    coordinator = ["Hi"]

    # Navigator
    conversation += "## 🛰️ Navigator\n"
    yield conversation

    for chunk in call_gemini():
        conversation += chunk
        yield conversation

    conversation += "\n\n"

    # Formation Planner
    conversation += "## 📐 Formation Planner\n"
    yield conversation

    for chunk in call_mistral():
        conversation += chunk
        yield conversation

    conversation += "\n\n"

    # Verifier
    conversation += "## ✅ Verifier\n"
    yield conversation

    for chunk in call_groq():
        conversation += chunk
        yield conversation

    conversation += "\n\n"

    # Coordinator
    conversation += "## 🎯 Coordinator\n"
    yield conversation

    for chunk in call_deepseek():
        conversation += chunk
        yield conversation

    conversation += "\n\n"


demo = gr.Interface(
    fn=run_swarm,
    inputs=[],
    outputs=gr.Markdown(),
    title="🤖 AI Swarm Simulation",
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "d:\MLProjects\GenAI\projects\Week2Projects\Swarm-Fighters\.venv\Lib\site-packages\gradio\queueing.py", line 957, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\MLProjects\GenAI\projects\Week2Projects\Swarm-Fighters\.venv\Lib\site-packages\gradio\route_utils.py", line 393, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\MLProjects\GenAI\projects\Week2Projects\Swarm-Fighters\.venv\Lib\site-packages\gradio\blocks.py", line 2280, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\MLProjects\GenAI\projects\Week2Projects\Swarm-Fighters\.venv\Lib\site-packages\gradio\blocks.py", line 1669, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\MLProjects\